In [10]:
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
import torch
from tqdm.auto import tqdm

import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[1] / '4_Baselines'))
import SASRec_class as sasrec

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

## 1. Load Processed Data

In [4]:
base_artifacts = Path.cwd().resolve().parents[2] / 'CausalI2I_artifacts'
data_sasrec = pd.read_csv(
    base_artifacts / 'Datasets' / 'Simulation' / 'data_sasrec.csv'
)

In [ ]:
L = 50
dropout = 0.5

In [6]:
unique_users = data_sasrec['user_id'].unique()
n_users = len(unique_users)

train_users = np.random.choice(
    unique_users, 
    size=int(0.8 * n_users), 
    replace=False)
test_users = np.setdiff1d(unique_users, train_users)

num_items = data_sasrec['item_id'].nunique()

In [7]:
users_dict = data_sasrec.groupby('user_id')['item_id'].apply(list).to_dict()
lens = [len(users_dict[user]) for user in users_dict]
np.mean(lens)

np.float64(342.56274834437085)

In [11]:
padding_idx = num_items

train_dataset = []
test_dataset = []
for user_id in tqdm(users_dict):
    users_dict[user_id] = [padding_idx] * (L - 2) + users_dict[user_id]
    i = 0
    while i + L <= len(users_dict[user_id]):
        if user_id in train_users:
            train_dataset.append(users_dict[user_id][i:i+L])
        else:
            test_dataset.append(users_dict[user_id][i:i+L])
        i += 1
train_dataset = np.array(train_dataset)
test_dataset = np.array(test_dataset)

  0%|          | 0/6040 [00:00<?, ?it/s]

# 2. Train the model

In [19]:
model = sasrec.SASRecTorch(
    num_items=num_items,
    max_seq_len=L,
    d_model=50,
    n_heads=1,
    n_layers=2,
    dropout=dropout,
    device="cuda",
)
model.fit(
    train_dataset=train_dataset,
    valid_dataset=test_dataset,
    batch_size=2**13,
    lr=1e-3,
    weight_decay=0.0, 
    num_epochs=20,
    )

Epoch | T-Loss | V-Loss | Pctl  | HR10  | NDCG  | Cosθ  | Elapsed Time
======|========|========|=======|=======|=======|=======|=============
    1 |  1.332 |  1.167 | 0.743 | 0.311 | 0.299 | None  |     01:15.7
    2 |  1.174 |  1.145 | 0.751 | 0.324 | 0.305 | 0.657 |     02:31.4
    3 |  1.152 |  1.136 | 0.756 | 0.329 | 0.308 | 0.759 |     03:47.0
    4 |  1.135 |  1.119 | 0.769 | 0.343 | 0.314 | 0.593 |     05:02.7
    5 |  1.125 |  1.116 | 0.770 | 0.346 | 0.315 | 0.590 |     06:18.5
    6 |  1.121 |  1.114 | 0.773 | 0.348 | 0.316 | 0.578 |     07:33.9
    7 |  1.113 |  1.101 | 0.779 | 0.360 | 0.322 | 0.409 |     08:49.6
    8 |  1.104 |  1.096 | 0.781 | 0.365 | 0.322 | 0.435 |     10:05.5
    9 |  1.100 |  1.094 | 0.783 | 0.368 | 0.324 | 0.464 |     11:21.1
   10 |  1.097 |  1.092 | 0.783 | 0.369 | 0.325 | 0.396 |     12:36.6
   11 |  1.094 |  1.088 | 0.785 | 0.375 | 0.328 | 0.374 |     13:52.2
   12 |  1.090 |  1.084 | 0.789 | 0.379 | 0.330 | 0.350 |     15:08.0
   13 |  1.088 |  

In [20]:
folder_path = base_artifacts / 'SASRec_Models'
model.save(path=folder_path / f'sasrec_simulation.pt')

init_dict = {
    "num_items": num_items,
    "max_seq_len": L,
    "d_model": model.d_model,
    "n_heads": model.n_heads,
    "n_layers": model.n_layers,
    "dropout": model.dropout,
    "device": model.device
}

with open(folder_path / f'sasrec_simulation_init_dict.pkl', 'wb') as f:
    pickle.dump(init_dict, f)


# 3. Load a Model

In [21]:
folder_path = base_artifacts / 'SASRec_Models'
with open(folder_path / f'sasrec_simulation_init_dict.pkl', 'rb') as f:
    init_dict_loaded = pickle.load(f)

loaded_model = sasrec.SASRecTorch(**init_dict_loaded)
loaded_model.load(folder_path / f'sasrec_simulation.pt')

Model loaded from /home/gouni/CausalI2I_artifacts/SASRec_Models/sasrec_simulation.pt.
num_items:     3952
max_seq_len:   50
device:        cuda
batch_size:    8192
lr:            0.001
weight_decay:  0.0
num_epochs:    20
saved_at:      2026-04-14 15:27:55
note:          None
